In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

# =========================
# 1. EXTRACT
# =========================
# load dataset (change file name if needed)
df = pd.read_csv("yellow_tripdata.csv")

print("Original Data Shape:", df.shape)
print(df.head())

# =========================
# 2. TRANSFORM (Cleaning)
# =========================

# convert datetime columns
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

# remove missing values
df = df.dropna()

# remove invalid passenger count
df = df[df["passenger_count"] > 0]

# remove zero or negative trip distance
df = df[df["trip_distance"] > 0]

# create new columns
df["trip_duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.day_name()

print("Cleaned Data Shape:", df.shape)

# =========================
# 3. LOAD (Database)
# =========================
conn = sqlite3.connect("taxi_data.db")
df.to_sql("trips", conn, if_exists="replace", index=False)

print("Data loaded into SQLite database!")

# =========================
# 4. ANALYSIS
# =========================

# Average fare & distance
avg_stats = df[["fare_amount", "trip_distance"]].mean()
print("\nAverage Fare & Distance:")
print(avg_stats)

# Demand by hour
demand_by_hour = df.groupby("pickup_hour").size()

# Top pickup locations
top_pickups = df["PULocationID"].value_counts().head(10)

# Most common routes
routes = df.groupby(["PULocationID", "DOLocationID"]).size().sort_values(ascending=False).head(10)

# Payment trends
payment = df["payment_type"].value_counts()

# Trip duration
avg_duration = df["trip_duration"].mean()
print("\nAverage Trip Duration (minutes):", avg_duration)

# =========================
# 5. VISUALIZATION
# =========================

# Demand by hour
plt.figure(figsize=(8,5))
demand_by_hour.plot(kind="line", marker="o")
plt.title("Taxi Demand by Hour")
plt.xlabel("Hour")
plt.ylabel("Trips")
plt.grid()
plt.show()

# Top pickup locations
plt.figure(figsize=(8,5))
top_pickups.plot(kind="bar", color="orange")
plt.title("Top Pickup Locations")
plt.show()

# Payment types
plt.figure(figsize=(6,4))
payment.plot(kind="pie", autopct="%1.1f%%")
plt.title("Payment Type Distribution")
plt.ylabel("")
plt.show()

# =========================
# 6. OPTIONAL: WEATHER MERGE
# =========================
# Example (if you have weather.csv)
"""
weather = pd.read_csv("weather.csv")
weather["date"] = pd.to_datetime(weather["date"])

df["date"] = df["tpep_pickup_datetime"].dt.date
weather["date"] = weather["date"].dt.date

merged = pd.merge(df, weather, on="date")

print("Merged Data:", merged.head())
"""

# =========================
# 7. OPTIMIZATION (SQL)
# =========================

query = """
SELECT pickup_hour, COUNT(*) as trip_count
FROM trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 5
"""

result = pd.read_sql(query, conn)
print("\nTop Busy Hours (SQL):")
print(result)

conn.close()

: 